# The sticky bead on a periodic sheet: the figures

The plots of `docs/bead_push_benchmark.md` section 9, made from the run matrix
of `analysis/sticky_bead_runs.py` (reservoir: box / global S0 / local S0 /
fixed; fluidity: frozen, nu = 0.05, 0.5, 5 per edge per us). Every figure is
drawn by `analysis/sticky_bead_analysis.py`; this notebook only loads the runs
once, writes each figure as **.svg and .png** into `<root>/analysis/figures`,
and shows it. The animations (GIF) are optional, section 6.

The runs are read from `SLIMED_STICKY_RUNS`, by default
`~/slimed_runs/sticky_bead/matrix`. Nothing is simulated here.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib as mpl
from IPython.display import Image, Markdown, display


def find_repo_root(start=None):
    """The SLIMED checkout this notebook sits in, wherever it is run from."""
    here = Path(start or Path.cwd()).resolve()
    for d in (here, *here.parents):
        if (d / "CMakeLists.txt").is_file() and (d / "analysis").is_dir():
            return d
    raise RuntimeError("not inside a SLIMED checkout")


REPO = find_repo_root()
sys.path.insert(0, str(REPO / "analysis"))
import sticky_bead_analysis as sb

ROOT = Path(os.environ.get("SLIMED_STICKY_RUNS", "~/slimed_runs/sticky_bead/matrix")).expanduser()
FIG = ROOT / "analysis" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
mpl.rcParams["savefig.dpi"] = 220
print("runs:", ROOT, "\nfigures:", FIG)

---
## 0. The runs

Every directory under the root with a bead log. Loading reads the control
net, the surface points and the per-face energies of all 901 frames of each
run, about a minute for the 13 runs.

In [ ]:
names = sorted(d.name for d in ROOT.iterdir() if (d / "inputbead.csv").exists())
runs = []
for name in names:
    try:
        runs.append(sb.StickyRun(str(ROOT / name)))
        r = runs[-1]
        print(f"{name:18s} {r.n_frames} frames to {r.time[-1]:.0f} us, {r.n_free} free vertices, tile {r.L0.round(1)}")
    except Exception as exc:
        print(f"skipping {name}: {exc}")
runs.sort(key=lambda r: (list(sb.RESERVOIR_STYLES).index(r.reservoir),
                         list(sb.FLUIDITY_COLORS).index(r.fluidity) if r.fluidity in sb.FLUIDITY_COLORS else 9))
stats = [sb.summarize(r) for r in runs]
t_end = min(r.time[-1] for r in runs)
t_stop = min(runs[0].t_stop, t_end)
vmax = max(np.percentile(r.bending_density(sb.frame_index(r, t_end)), 99) for r in runs)


def figure(writer, name, *args, **kwargs):
    """Write one figure as .svg and .png (the writer builds it from the runs) and show the .png."""
    for ext in ("svg", "png"):
        writer(runs, str(FIG / f"{name}.{ext}"), *args, **kwargs)
    display(Image(filename=str(FIG / f"{name}.png")))

---
## 1. Traces

Force on the bead, bonded fraction, tension and the reservoir's area; grey
while the bead moves, white while it holds.

In [ ]:
figure(sb.write_traces, "traces")

## 2. Relaxation after the stop

Exponential fits of the force, the bonded count and the tension over the
hold phase.

In [ ]:
figure(sb.write_hold_relaxation, "hold_relaxation", stats)

## 3. Bending energy maps

Top view of the control net at the stop and at the end. Only the
energy-carrying faces are drawn, tiled by the period for the margin; the
mesh's image-band copies go stale under flips and are left out (see the
docs).

In [ ]:
for t_us in sorted({t_stop, t_end}):
    figure(sb.energy_map_grid, f"energy_maps_{t_us:.0f}us", t_us, vmax=vmax)

## 4. Ring profiles

Mean bending energy density in rings around the bead's axis.

In [ ]:
figure(sb.write_profiles, "energy_profiles", sorted({t_stop, t_end}))

## 5. Fourier analysis of the hold phase

The static imprint of the bead and the fluctuation spectrum of the sheet
outside its footprint, with the kc and sigma fits.

In [ ]:
figure(sb.write_spectra, "spectra", stats)

## 6. The numbers

`stats.md` beside the figures; the table `docs/bead_push_benchmark.md`
section 9.2 is condensed from it.

In [ ]:
sb.write_stats(stats, str(FIG / "stats.md"), str(FIG / "stats.csv"))
display(Markdown((FIG / "stats.md").read_text()))

## 7. Animations (optional)

One GIF per run and a four-panel comparison, about a minute each. Set
`MAKE_GIFS = True` to render them.

In [ ]:
MAKE_GIFS = False
COMPARE = ["box_frozen", "box_nu0.05", "box_nu0.5", "box_nu5"]
if MAKE_GIFS:
    for r in runs:
        n = sb.write_animation([r], str(FIG / f"anim_{r.name}.gif"), 150, 15, vmax)
        print(f"anim_{r.name}.gif: {n} frames")
    chosen = [r for r in runs if r.name in COMPARE][:4]
    if chosen:
        sb.write_animation(chosen, str(FIG / "compare.gif"), 150, 15, vmax)
        print("compare.gif written")